# Regression, Regularisation & Sparse Feature Selection

Hands-on practice with three core supervised-learning ideas in scikit-learn:

1. **K-Nearest Neighbours regression**: how `k`, distance weighting and the distance metric change performance (California Housing).
2. **Linear models with regularisation**: comparing OLS, Lasso (L1), Ridge (L2) and ElasticNet on a mixed categorical/numeric dataset (UCI Student Performance).
3. **L1-regularised logistic regression for feature selection**: classifying a high-dimensional gene-expression dataset (~21k features, 172 samples) and seeing how the regularisation strength controls how many features survive.

Across all three, the same habits apply: split before preprocessing, fit scalers/encoders on the training set only, and choose a metric that fits the task (R² for regression, accuracy for classification).

In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet, LogisticRegression
from sklearn.metrics import r2_score, accuracy_score

## 1. KNN Regression: California Housing

The California Housing dataset (1990 US census, one row per block group) has 20,640 samples and 8 numeric features. The target is the median house value in units of $100k.

KNN relies on distances, so **feature scaling is essential**: without it, features with large ranges (for example population) dominate the distance. The model is scored with the **coefficient of determination (R²)**.

In [2]:
california = datasets.fetch_california_housing()
print('shape:', california.data.shape)
print('features:', california.feature_names)
print('target:', california.target_names)

X, y = california.data, california.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

shape: (20640, 8)
features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
target: ['MedHouseVal']


### Effect of the number of neighbours `k`

In [3]:
for k in [3, 30, 300]:
    knn = KNeighborsRegressor(n_neighbors=k).fit(X_train_s, y_train)
    print(f'k={k:>3}: R² = {r2_score(y_test, knn.predict(X_test_s)):.4f}')

k=  3: R² = 0.6439


k= 30: R² = 0.6710


k=300: R² = 0.6028


### Distance-weighted KNN with different metrics (k = 30)

In [4]:
for metric in ['euclidean', 'cosine', 'manhattan']:
    knn = KNeighborsRegressor(n_neighbors=30, weights='distance', metric=metric).fit(X_train_s, y_train)
    print(f'{metric:>9}: R² = {r2_score(y_test, knn.predict(X_test_s)):.4f}')

euclidean: R² = 0.6770


   cosine: R² = 0.6704


manhattan: R² = 0.7127


**Takeaways**
- Small `k` (3) gives high variance and overfits local noise. Very large `k` (300) oversmooths and underfits. A moderate `k` works best.
- Distance weighting lets closer neighbours count more, which usually helps over uniform weights at the same `k`.
- **Manhattan** distance tends to work well here because it is less dominated by a single large feature difference. **Cosine** ignores magnitude, which loses useful information for this kind of tabular data.

## 2. Regularised Linear Regression: Student Performance

[UCI Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance) (maths course): 395 students and 32 input attributes covering demographic, social and school-related information, plus the final grade `G3` as the target.

Many columns are categorical, so they are one-hot encoded. The encoder output for the test set is **aligned to the training columns**, so a category seen only in one split cannot misalign the feature matrix.

In [5]:
student = pd.read_csv('../data/student-mat.csv')
print('dataset shape:', student.shape)

X = student.drop(columns=['G3'])
y = student['G3']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

X_train_enc = pd.get_dummies(X_train)
X_test_enc = pd.get_dummies(X_test).reindex(columns=X_train_enc.columns, fill_value=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_enc)
X_test_s = scaler.transform(X_test_enc)

dataset shape: (395, 33)
X_train: (316, 32) | X_test: (79, 32)


In [6]:
models = {
    'LinearRegression': LinearRegression(),
    'Lasso (L1)': Lasso(),
    'Ridge (L2)': Ridge(),
    'ElasticNet (L1+L2)': ElasticNet(),
}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    r2 = r2_score(y_test, model.predict(X_test_s))
    nonzero = int(np.sum(model.coef_ != 0))
    print(f'{name:<20} R² = {r2:.4f} | non-zero coefficients = {nonzero}/{X_train_s.shape[1]}')

LinearRegression     R² = 0.7332 | non-zero coefficients = 58/58
Lasso (L1)           R² = 0.7376 | non-zero coefficients = 2/58
Ridge (L2)           R² = 0.7336 | non-zero coefficients = 58/58
ElasticNet (L1+L2)   R² = 0.6978 | non-zero coefficients = 3/58


**Takeaways**
- `G1` and `G2` (earlier-period grades) are very strong predictors of `G3`, so even plain OLS does well.
- **Lasso** drives most coefficients to exactly zero and keeps only the most informative features. This acts as built-in feature selection and often generalises better than OLS.
- **Ridge** shrinks coefficients but keeps all of them. **ElasticNet** sits between the two.

## 3. Sparse Logistic Regression on Gene-Expression Data

A high-dimensional classification problem: **172 samples and ~21,000 gene-expression features**, with a binary label `ClassType`. When features far outnumber samples (p ≫ n), an unregularised model would overfit badly. **L1 regularisation** forces most coefficients to zero and in effect selects a small subset of genes.

> The dataset file is not included in this repo (see `data/README.md`). Place `GenomicData.csv` in `data/` to run this section.

In [7]:
df = pd.read_csv('../data/GenomicData.csv')
print('original shape:', df.shape)

df = df.drop(df.columns[[0, 1]], axis=1)   # index + sample-ID columns carry no signal
print('after dropping ID columns:', df.shape)

np.random.seed(42)
print('rows with any NaN:', df.isna().any(axis=1).sum(), '| columns with any NaN:', df.isna().any(axis=0).sum())
print(df['ClassType'].value_counts())

original shape: (172, 21053)
after dropping ID columns: (172, 21051)


rows with any NaN: 16 | columns with any NaN: 1882
ClassType
1    106
3     66
Name: count, dtype: int64


### Handling missing values: drop rows or drop columns?

Only a few samples contain NaNs, but those NaNs are spread across ~1,900 gene columns. Each approach has a cost:
- **Drop rows:** keeps every feature but loses samples, which are already scarce.
- **Drop columns:** keeps every sample and loses about 9% of the features, which barely matters when there are 19k+ others.

Both approaches are tried below with C = 0.05 and C = 10 (smaller C means stronger regularisation).

In [8]:
def fit_l1_logreg(data, label):
    y = data['ClassType']
    X = data.drop(columns=['ClassType'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_tr, X_te = scaler.fit_transform(X_tr), scaler.transform(X_te)
    print(f'--- {label}: {data.shape[0]} samples, {X.shape[1]} features ---')
    for C in [0.05, 10]:
        model = LogisticRegression(l1_ratio=1, solver='liblinear', C=C, random_state=42).fit(X_tr, y_tr)  # l1_ratio=1 -> pure L1
        acc = accuracy_score(y_te, model.predict(X_te))
        print(f'C={C:<5} test accuracy = {acc:.4f} | genes selected = {int(np.sum(model.coef_ != 0))}')

fit_l1_logreg(df.dropna(axis=0), 'drop rows with NaN')
fit_l1_logreg(df.dropna(axis=1), 'drop columns with NaN')

--- drop rows with NaN: 156 samples, 21050 features ---
C=0.05  test accuracy = 0.8718 | genes selected = 3


C=10    test accuracy = 0.9487 | genes selected = 236


--- drop columns with NaN: 172 samples, 19168 features ---
C=0.05  test accuracy = 0.9302 | genes selected = 10


C=10    test accuracy = 0.9767 | genes selected = 185


**Takeaways**
- With strong regularisation (**C = 0.05**), only **3 to 10 genes** out of ~20k get non-zero weights and the model still reaches **87 to 93%** test accuracy. That is a very compact, interpretable model.
- With weak regularisation (**C = 10**), **185 to 236 genes** are kept and accuracy rises to **95 to 98%**. This is the trade-off between sparsity/interpretability and raw accuracy.
- Dropping the **columns** with missing values beat dropping the **rows** at both values of C. With p ≫ n, each sample is far more valuable than any single gene.

## Summary of skills practised

- Train/test splitting and preventing **data leakage** (fit preprocessing on training data only)
- **Feature scaling** with `StandardScaler` and why distance-based models depend on it
- **One-hot encoding** categorical data while keeping train/test columns aligned
- **KNN regression** hyperparameters: `k`, distance weighting and metric choice
- **L1 / L2 / ElasticNet regularisation** and how each affects the coefficients
- **Sparse logistic regression** for feature selection in high-dimensional (p ≫ n) data
- Missing-data strategies and their trade-offs